In [24]:
import chromadb
from sentence_transformers import SentenceTransformer


# =========================================================
# 1. 기본 설정
# =========================================================

EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"

# ChromaDB를 만들 때 사용한 실제 경로와 동일하게 설정
CHROMA_PATH = "../chroma_db"

# ChromaDB를 만들 때 사용한 collection 이름
COLLECTION_NAME = "maplestory_guides"


# =========================================================
# 2. 임베딩 모델 / ChromaDB 연결
# =========================================================

embed_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

collection = client.get_collection(
    name=COLLECTION_NAME
)

print("Collection:", collection.name)
print("문서 수:", collection.count())


# =========================================================
# 3. 질문 분류를 위한 키워드
# =========================================================

JOB_KEYWORDS = [
    # 직업 관련 일반 단어
    "직업",
    "직업추천",
    "직업 추천",
    "주스탯",
    "주 스탯",
    "주스텟",
    "무기",
    "전직",

    # 직업군
    "전사",
    "마법사",
    "궁수",
    "도적",
    "해적",

    # 예시 직업명
    "히어로",
    "팔라딘",
    "다크나이트",
    "아크메이지",
    "비숍",
    "보우마스터",
    "신궁",
    "패스파인더",
    "나이트로드",
    "섀도어",
    "듀얼블레이드",
    "바이퍼",
    "캡틴",
    "캐논슈터",

    "소울마스터",
    "미하일",
    "플레임위자드",
    "윈드브레이커",
    "나이트워커",
    "스트라이커",

    "아란",
    "에반",
    "메르세데스",
    "팬텀",
    "루미너스",
    "은월",

    "데몬슬레이어",
    "데몬어벤져",
    "블래스터",
    "배틀메이지",
    "와일드헌터",
    "메카닉",
    "제논",

    "카이저",
    "카인",
    "카데나",
    "엔젤릭버스터",

    "아델",
    "일리움",
    "아크",
    "칼리",

    "호영",
    "라라",

    "제로",
    "키네시스",
    "렌",
]


ITEM_KEYWORDS = [
    "확률",
    "아이템",
    "장비",
    "큐브",
    "잠재",
    "잠재능력",
    "에디셔널",
    "스타포스",
    "강화",
    "획득 확률",
]


# =========================================================
# 4. 질문에 따라 검색 범위 결정
# =========================================================

def get_search_filter(query):
    """
    사용자 질문을 간단한 키워드 기반으로 분류하여
    검색할 source를 결정합니다.

    job 질문  -> {"source": "job"}
    item 질문 -> {"source": "item"}
    그 외     -> None (전체 문서 검색)
    """

    query = query.strip().lower()

    # 직업 관련 질문
    if any(keyword.lower() in query for keyword in JOB_KEYWORDS):
        return {"source": "job"}

    # 아이템 / 확률 관련 질문
    if any(keyword.lower() in query for keyword in ITEM_KEYWORDS):
        return {"source": "item"}

    # 분류가 확실하지 않을 경우 전체 검색
    return None


# =========================================================
# 5. Top-K Retriever
# =========================================================

def retrieve_top_k(
    query,
    collection,
    embed_model,
    top_k=5,
    where=None,
):
    query = query.strip()

    if not query:
        raise ValueError("검색 질문이 비어 있습니다.")

    if top_k < 1:
        raise ValueError("top_k는 1 이상이어야 합니다.")

    document_count = collection.count()

    if document_count == 0:
        return []

    # DB 문서와 동일한 임베딩 모델 사용
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True,
    )

    if query_embedding.ndim != 2:
        raise ValueError(
            f"질문 임베딩 차원 오류: {query_embedding.shape}"
        )

    if query_embedding.shape[0] != 1:
        raise ValueError(
            f"질문은 1개여야 합니다: {query_embedding.shape}"
        )

    query_kwargs = {
        "query_embeddings": query_embedding.tolist(),
        "n_results": min(top_k, document_count),
        "include": [
            "documents",
            "metadatas",
            "distances",
        ],
    }

    # source filtering
    if where:
        query_kwargs["where"] = where

    raw_results = collection.query(**query_kwargs)

    results = []

    for index, chunk_id in enumerate(
        raw_results["ids"][0]
    ):
        distance = raw_results["distances"][0][index]

        results.append(
            {
                "rank": index + 1,
                "id": chunk_id,
                "page_content": (
                    raw_results["documents"][0][index]
                ),
                "metadata": (
                    raw_results["metadatas"][0][index]
                ),
                "distance": distance,
                "score": 1.0 - distance,
            }
        )

    return results


# =========================================================
# 6. Router + Retriever
# =========================================================

def search_documents(
    query,
    collection,
    embed_model,
    top_k=5,
):
    """
    질문을 분류한 후 적절한 source에서 Top-K 검색
    """

    where = get_search_filter(query)

    print("=" * 80)
    print(f"질문: {query}")

    if where is None:
        print("검색 범위: 전체 문서")
    else:
        print(f"검색 범위: {where['source']}")

    print("=" * 80)

    results = retrieve_top_k(
        query=query,
        collection=collection,
        embed_model=embed_model,
        top_k=top_k,
        where=where,
    )

    return results


# =========================================================
# 7. 검색 테스트
# =========================================================

query = "히어로가 사용하는 무기는 뭐야?"

results = search_documents(
    query=query,
    collection=collection,
    embed_model=embed_model,
    top_k=5,
)


# =========================================================
# 8. 결과 출력
# =========================================================

for result in results:
    print("=" * 80)

    print(f"순위: {result['rank']}")
    print(f"점수: {result['score']:.4f}")

    print(
        "source:",
        result["metadata"].get("source")
    )

    print(
        "문서명:",
        result["metadata"].get("name")
    )

    print(
        "URL:",
        result["metadata"].get("url")
    )

    print()

    print(result["page_content"])

    print()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2147.78it/s]


NotFoundError: Collection [maplestory_guides] does not exist